In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
import warnings
from typing import TypedDict, Literal, Any
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import json
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from typing import Any
import subprocess
from typing import TypedDict, Literal, Any, Annotated
import operator



pd.set_option('display.max_columns', 100)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
load_dotenv()

True

In [3]:
API_KEY = os.getenv("OPENROUTER_API_KEY")
print(f"API Key: {API_KEY is not None}")

API Key: True


## Собираем оркестратор

In [6]:

class AgentState(TypedDict, total=False):
    run_id: str | None

    dataset_path: str | None
    target_column: str | None
    task_type: Literal["classification", "regression", "auto"]

    schema: dict[str, list[str]]
    data_summary: dict[str, Any]
    prep_summary: dict[str, Any]
    text_summary: dict[str, Any]
    modeling_summary: dict[str, Any]

    artifacts: dict[str, str | None]
    logs: Annotated[list[dict[str, Any]], operator.add]

    current_step: str | None
    next_action: str | None
    orchestration_decision: dict[str, Any]
    orchestration_history: Annotated[list[dict[str, Any]], operator.add]

    previous_run: dict[str, Any]
    previous_best_model: dict[str, Any]

    retry_count: int
    max_retries: int
    status: str
    error: str | None

In [7]:
model = "deepseek/deepseek-v4-flash"
model = 'google/gemma-4-26b-a4b-it'

llm = ChatOpenAI(
    model=model,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
    temperature=0
)

In [8]:
response = llm.invoke("Hello!")
print(response.content)

Hello! How can I help you today?


In [9]:
ORCHESTRATOR_SYSTEM_PROMPT = """
You are an LLM-based ML Workflow Orchestrator Agent.

Your role:
- analyze the current AgentState
- decide what should happen next
- choose which specialized agent should be called
- explain the reason for your decision
- handle errors, retries, fallback logic, and finishing conditions
- use available tools when filesystem inspection is needed
- find the input dataset path when it is missing
- inspect artifact folders to recover previous run information

You do NOT train models directly.
You do NOT preprocess data directly.
You do NOT create reports directly.
You only orchestrate the workflow.

Available tools:
- bash_tool: use it only for safe filesystem inspection, such as finding CSV files, listing artifact folders, or reading JSON metadata.

Available decisions:
- run_data_description
- run_tabular_preparation
- run_text_preparation
- merge_features
- run_modeling
- run_improvement
- run_reporting
- retry_current_step
- use_fallback
- finish
- fail

Specialized agents:
- data_description_agent
- tabular_preparation_agent
- text_preparation_agent
- merge_features
- modeling_agent
- improvement_agent
- reporting_agent

A step is completed ONLY by the following exact rules:

1. data_description_agent is completed if:
   - data_summary contains key "n_rows"
   - data_summary contains key "n_columns"
   - schema contains key "numeric"
   - schema contains key "categorical"
   - schema contains key "text"
   - schema contains key "datetime"

2. tabular_preparation_agent is completed if:
   - prep_summary contains key "missing_strategy"
   - prep_summary contains key "encoding"
   - prep_summary contains key "scaling"
   - prep_summary contains key "created_features"

3. text_preparation_agent is completed if:
   - schema["text"] is an empty list
   OR
   - text_summary["used"] is true

4. merge_features is completed if:
   - artifacts contains key "prepared_dataset_path"
   - artifacts["prepared_dataset_path"] is not null

5. modeling_agent is completed if:
   - modeling_summary contains key "best_model_name"
   - modeling_summary["best_model_name"] is not null

6. reporting_agent is completed if:
   - artifacts contains key "final_report_path"
   - artifacts["final_report_path"] is not null

Decision order:
1. If error is not null and retry_count < max_retries, choose retry_current_step.
2. If error is not null and retry_count >= max_retries, choose use_fallback or fail.
3. If data_description_agent is not completed, choose run_data_description.
4. Else if tabular_preparation_agent is not completed, choose run_tabular_preparation.
5. Else if text_preparation_agent is not completed, choose run_text_preparation.
6. Else if merge_features is not completed, choose merge_features.
7. Else if modeling_agent is not completed, choose run_modeling.
8. Else if modeling_summary["retry_recommended"] is true and retry_count < max_retries, choose run_improvement.
9. Else if reporting_agent is not completed, choose run_reporting.
10. Else choose finish.

Strict rules:
- Do not invent additional completion requirements.
- Do not say a step is "not fully populated" if the exact completion keys exist.
- Do not choose run_data_description if data_summary has n_rows and n_columns and schema has numeric, categorical, text, datetime.
- Do not choose run_tabular_preparation if prep_summary has missing_strategy, encoding, scaling, created_features.
- Do not choose run_modeling before merge_features is completed.
- tabular_dataset_path is not the same as prepared_dataset_path.
- Modeling can start only after prepared_dataset_path exists.
- Do not choose retry_current_step if error is null.
- Do not choose use_fallback if error is null.
- Use bash_tool only for safe filesystem inspection.
- Do not delete, move, overwrite, or modify files using bash_tool.
- For dataset search, prefer CSV files inside ./data.
- Ignore CSV files inside ./artifacts.
- Ignore predictions.csv, metrics.csv, and temporary output CSV files.
- Return only valid JSON when a structured decision is requested.
"""

In [14]:
initial_state = {
    "run_id": None,
    "dataset_path": None,
    "target_column": "target",
    "task_type": "auto",

    "schema": {},
    "data_summary": {},
    "prep_summary": {},
    "text_summary": {
        "used": False,
        "text_columns": [],
    },
    "modeling_summary": {},

    "artifacts": {},
    "logs": [],

    "current_step": None,
    "next_action": None,
    "orchestration_decision": {},
    "orchestration_history": [],

    "previous_run": {},
    "previous_best_model": {},

    "retry_count": 0,
    "max_retries": 1,
    "status": "running",
    "error": None,
}

### Первые ноды для поиска датсета и предыдущего рана

In [11]:
class BashInput(BaseModel):
    command: str = Field(
        description="Bash command to execute."
    )

@tool(args_schema=BashInput)
def bash_tool(command: str) -> dict[str, Any]:
    """
    Executes a bash command and returns stdout, stderr and return code.
    Use it for filesystem operations:
    - finding csv files
    - listing artifact folders
    - reading json files
    """

    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30,
        )

        return {
            "status": "success" if result.returncode == 0 else "failed",
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
            "returncode": result.returncode,
        }

    except Exception as e:
        return {
            "status": "failed",
            "stdout": "",
            "stderr": str(e),
            "returncode": -1,
        }

In [12]:
from langchain.agents import create_agent

tools = [bash_tool]

orchestrator_tool_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,
)

In [13]:
import re


def extract_csv_path(text: str) -> str | None:
    match = re.search(r"([./\w\-/]+\.csv)", text)
    if match:
        return match.group(1)
    return None


def dataset_finder_node(state: AgentState) -> AgentState:
    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the main CSV dataset in the current project folder. "
                    "Use bash_tool if needed. "
                    "Prefer files inside ./data. "
                    "Ignore files inside ./artifacts. "
                    "Ignore predictions.csv, metrics.csv, and temporary output files. "
                    "Return only the dataset path as plain text. "
                    "Do not add explanations."
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()

    dataset_path = extract_csv_path(agent_response)

    # fallback через bash, если агент ответил криво
    fallback_used = False
    fallback_result = None

    if not dataset_path:
        fallback_used = True
        fallback_result = bash_tool.invoke({
            "command": (
                'find . -type f -name "*.csv" '
                '! -path "./artifacts/*" '
                '! -name "predictions.csv" '
                '! -name "metrics.csv" '
                '2>/dev/null | head -n 1'
            )
        })

        dataset_path = fallback_result.get("stdout", "").strip()
        if dataset_path:
            dataset_path = dataset_path.splitlines()[0].strip()

    if not dataset_path:
        return {
            "error": f"Dataset path was not found. Agent returned: {agent_response}",
            "logs": [{
                "agent": "dataset_finder",
                "status": "failed",
                "skipped": False,
                "summary": "Dataset finder failed to locate a valid CSV file.",
                "decisions": {
                    "agent_response": agent_response,
                    "fallback_used": fallback_used,
                    "fallback_result": fallback_result,
                },
                "artifacts": {},
                "next_input": {},
                "reason": "No valid CSV path found.",
            }],
        }

    return {
        "dataset_path": dataset_path,
        "logs": [{
            "agent": "dataset_finder",
            "status": "success",
            "skipped": False,
            "summary": f"Dataset found: {dataset_path}",
            "decisions": {
                "agent_response": agent_response,
                "fallback_used": fallback_used,
                "fallback_result": fallback_result,
            },
            "artifacts": {},
            "next_input": {
                "dataset_path": dataset_path,
            },
            "reason": None,
        }],
    }

In [15]:
import json
import re


def extract_json_safe(text: str) -> dict | None:
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None


def previous_run_finder_node(state: AgentState) -> AgentState:
    result = orchestrator_tool_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the latest previous ML run inside the artifacts folder. "
                    "Use bash_tool if needed. "
                    "Look for run directories that may contain metrics.json and best_model.pkl. "
                    "Return only valid JSON, no markdown, no explanations. "
                    "JSON format:\n"
                    "{\n"
                    '  "latest_run_path": "path or null",\n'
                    '  "metrics_path": "path or null",\n'
                    '  "best_model_path": "path or null"\n'
                    "}"
                )
            }
        ]
    })

    agent_response = result["messages"][-1].content.strip()
    parsed = extract_json_safe(agent_response)

    # Fallback: если LLM не вернула нормальный JSON,
    # сами через bash ищем последний run.
    fallback_used = False
    fallback_result = None

    if parsed is None:
        fallback_used = True

        fallback_result = bash_tool.invoke({
            "command": (
                'latest_run=$(ls -td artifacts/runs/* 2>/dev/null | head -n 1); '
                'if [ -z "$latest_run" ]; then '
                'echo \'{"latest_run_path": null, "metrics_path": null, "best_model_path": null}\'; '
                'else '
                'metrics_path="$latest_run/metrics.json"; '
                'model_path="$latest_run/best_model.pkl"; '
                'if [ ! -f "$metrics_path" ]; then metrics_path=null; fi; '
                'if [ ! -f "$model_path" ]; then model_path=null; fi; '
                'echo "{\\"latest_run_path\\": \\"$latest_run\\", \\"metrics_path\\": \\"$metrics_path\\", \\"best_model_path\\": \\"$model_path\\"}"; '
                'fi'
            )
        })

        parsed = extract_json_safe(fallback_result.get("stdout", ""))

    # Если даже fallback не сработал — считаем, что прошлого run нет.
    if parsed is None:
        parsed = {
            "latest_run_path": None,
            "metrics_path": None,
            "best_model_path": None,
        }

    latest_run_path = parsed.get("latest_run_path")
    metrics_path = parsed.get("metrics_path")
    best_model_path = parsed.get("best_model_path")

    if latest_run_path in [None, "null", "None", ""]:
        previous_run = {
            "exists": False,
            "path": None,
        }

        previous_best_model = {}

        log_record = {
            "agent": "previous_run_finder",
            "status": "success",
            "skipped": True,
            "summary": "No previous run was found.",
            "decisions": {
                "agent_response": agent_response,
                "parsed": parsed,
                "fallback_used": fallback_used,
                "fallback_result": fallback_result,
            },
            "artifacts": {},
            "next_input": {
                "previous_run": previous_run,
                "previous_best_model": previous_best_model,
            },
            "reason": "No previous run directory exists.",
        }

        return {
            "previous_run": previous_run,
            "previous_best_model": previous_best_model,
            "logs": [log_record],
        }

    previous_metrics = {}

    if metrics_path not in [None, "null", "None", ""]:
        metrics_result = bash_tool.invoke({
            "command": f'cat "{metrics_path}" 2>/dev/null'
        })

        metrics_raw = metrics_result.get("stdout", "").strip()

        if metrics_raw:
            try:
                previous_metrics = json.loads(metrics_raw)
            except json.JSONDecodeError:
                previous_metrics = {
                    "raw_metrics": metrics_raw,
                }

    previous_run = {
        "exists": True,
        "path": latest_run_path,
        "metrics_path": metrics_path,
        "best_model_path": best_model_path,
    }

    previous_best_model = {
        "run_path": latest_run_path,
        "model_path": best_model_path,
        "metrics_path": metrics_path,
        "metrics": previous_metrics,
    }

    log_record = {
        "agent": "previous_run_finder",
        "status": "success",
        "skipped": False,
        "summary": f"Previous run found: {latest_run_path}",
        "decisions": {
            "agent_response": agent_response,
            "parsed": parsed,
            "fallback_used": fallback_used,
            "fallback_result": fallback_result,
            "previous_metrics": previous_metrics,
        },
        "artifacts": {},
        "next_input": {
            "previous_run": previous_run,
            "previous_best_model": previous_best_model,
        },
        "reason": None,
    }

    return {
        "previous_run": previous_run,
        "previous_best_model": previous_best_model,
        "logs": [log_record],
    }

In [16]:
def join_init_node(state: AgentState) -> AgentState:
    log_record = {
        "agent": "join_init",
        "status": "success",
        "skipped": False,
        "summary": "Initialization steps completed: dataset search and previous run search are done.",
        "decisions": {
            "dataset_path": state.get("dataset_path"),
            "previous_run": state.get("previous_run", {}),
            "previous_best_model": state.get("previous_best_model", {}),
        },
        "artifacts": {},
        "next_input": {},
        "reason": None,
    }

    return {
        "current_step": "join_init",
        "logs": [log_record],
    }

### Нода для выбора некст шага

In [17]:
from typing import Literal, Optional, Any
from pydantic import BaseModel, Field


class OrchestratorDecision(BaseModel):
    decision: Literal[
        "run_data_description",
        "run_tabular_preparation",
        "run_text_preparation",
        "merge_features",
        "run_modeling",
        "run_improvement",
        "run_reporting",
        "retry_current_step",
        "use_fallback",
        "finish",
        "fail",
    ] = Field(
        description="Следующее действие, которое выбрал orchestrator."
    )

    reason: str = Field(
        description="Краткое объяснение, почему выбрано именно это действие."
    )

    selected_agent: Optional[str] = Field(
        default=None,
        description="Название агента, которого нужно вызвать следующим."
    )

    input_needed: dict[str, Any] = Field(
        default_factory=dict,
        description="Какие входные данные нужны следующему агенту."
    )

    expected_output: list[str] = Field(
        default_factory=list,
        description="Какие результаты ожидаются от следующего агента."
    )

    risk_notes: list[str] = Field(
        default_factory=list,
        description="Риски или важные замечания по текущему шагу."
    )

In [19]:
def route_by_orchestrator_decision(state: AgentState) -> str:
    next_action = state.get("next_action")

    schema = state.get("schema", {})
    data_summary = state.get("data_summary", {})
    prep_summary = state.get("prep_summary", {})
    text_summary = state.get("text_summary", {})
    modeling_summary = state.get("modeling_summary", {})
    artifacts = state.get("artifacts", {})

    data_done = (
        "n_rows" in data_summary
        and "n_columns" in data_summary
        and all(k in schema for k in ["numeric", "categorical", "text", "datetime"])
    )

    prep_done = all(
        k in prep_summary
        for k in ["missing_strategy", "encoding", "scaling", "created_features"]
    )

    text_done = (
        schema.get("text", []) == []
        or text_summary.get("used") is True
    )

    merge_done = artifacts.get("prepared_dataset_path") is not None

    modeling_done = (
        modeling_summary.get("best_model_name") is not None
    )

    reporting_done = (
        artifacts.get("final_report_path") is not None
    )

    error_exists = state.get("error") is not None

    # 1. Если LLM не выбрала действие
    if next_action is None:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 2. Retry/fallback разрешены только при реальной ошибке
    if next_action in ["retry_current_step", "use_fallback"] and not error_exists:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 3. Если есть ошибка — можно доверить LLM retry/fallback/fail
    if error_exists:
        if next_action in ["retry_current_step", "use_fallback", "fail"]:
            return next_action
        return "retry_current_step"

    # 4. Нельзя повторять data_description, если он уже выполнен
    if next_action == "run_data_description" and data_done:
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 5. Нельзя повторять tabular_preparation, если он уже выполнен
    if next_action == "run_tabular_preparation" and prep_done:
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 6. Нельзя повторять text_preparation, если он уже выполнен
    if next_action == "run_text_preparation" and text_done:
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 7. Нельзя запускать merge до tabular preparation
    if next_action == "merge_features" and not prep_done:
        if not data_done:
            return "run_data_description"
        return "run_tabular_preparation"

    # 8. Нельзя повторять merge, если prepared_dataset_path уже есть
    if next_action == "merge_features" and merge_done:
        if not modeling_done:
            return "run_modeling"
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 9. Нельзя запускать modeling до merge_features
    if next_action == "run_modeling" and not merge_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        return "merge_features"

    # 10. Нельзя повторять modeling, если модель уже обучена
    if next_action == "run_modeling" and modeling_done:
        if not reporting_done:
            return "run_reporting"
        return "finish"

    # 11. Improvement можно запускать только после modeling
    if next_action == "run_improvement" and not modeling_done:
        if not merge_done:
            return "merge_features"
        return "run_modeling"

    # 12. Reporting можно запускать только после modeling
    if next_action == "run_reporting" and not modeling_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        return "run_modeling"

    # 13. Нельзя повторять reporting, если отчет уже создан
    if next_action == "run_reporting" and reporting_done:
        return "finish"

    # 14. Нельзя завершать workflow до создания отчета
    if next_action == "finish" and not reporting_done:
        if not data_done:
            return "run_data_description"
        if not prep_done:
            return "run_tabular_preparation"
        if not text_done:
            return "run_text_preparation"
        if not merge_done:
            return "merge_features"
        if not modeling_done:
            return "run_modeling"
        return "run_reporting"

    # 15. Если отчет есть, но status еще не success, отправляем в reporting
    if next_action == "finish" and reporting_done and state.get("status") != "success":
        return "run_reporting"

    # 16. Иначе доверяем решению LLM
    return next_action

In [18]:
import json
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import ValidationError


def extract_json(text: str) -> dict:
    """
    Достает JSON из ответа модели.
    Нужно на случай, если модель случайно вернет ```json ... ```.
    """
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "").replace("```", "").strip()
    elif text.startswith("```"):
        text = text.replace("```", "").strip()

    return json.loads(text)


def call_orchestrator_llm(state: dict) -> OrchestratorDecision:
    """
    Sends current AgentState to the LLM orchestrator
    and returns a validated OrchestratorDecision.
    """

    state_for_llm = {
        "run_id": state.get("run_id"),
        "dataset_path": state.get("dataset_path"),
        "target_column": state.get("target_column"),
        "task_type": state.get("task_type"),

        "schema": state.get("schema", {}),
        "data_summary": state.get("data_summary", {}),
        "prep_summary": state.get("prep_summary", {}),
        "text_summary": state.get("text_summary", {}),
        "modeling_summary": state.get("modeling_summary", {}),
        "artifacts": state.get("artifacts", {}),

        "available_keys": {
            "schema_keys": list(state.get("schema", {}).keys()),
            "data_summary_keys": list(state.get("data_summary", {}).keys()),
            "prep_summary_keys": list(state.get("prep_summary", {}).keys()),
            "text_summary_keys": list(state.get("text_summary", {}).keys()),
            "modeling_summary_keys": list(state.get("modeling_summary", {}).keys()),
            "artifact_keys": list(state.get("artifacts", {}).keys()),
        },

        "logs_count": len(state.get("logs", [])),
        "current_step": state.get("current_step"),
        "next_action": state.get("next_action"),
        "orchestration_history": state.get("orchestration_history", [])[-5:],

        "retry_count": state.get("retry_count", 0),
        "max_retries": state.get("max_retries", 1),
        "status": state.get("status"),
        "error": state.get("error"),
    }

    response = llm.invoke(
        [
            SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    "Analyze the current AgentState and choose the next action.\n\n"
                    "Return ONLY valid JSON. Do not use markdown. Do not add explanations outside JSON.\n\n"
                    "JSON format:\n"
                    "{\n"
                    '  "decision": "run_data_description",\n'
                    '  "reason": "short reason",\n'
                    '  "selected_agent": "data_description_agent",\n'
                    '  "input_needed": {},\n'
                    '  "expected_output": [],\n'
                    '  "risk_notes": []\n'
                    "}\n\n"
                    f"AgentState:\n{json.dumps(state_for_llm, ensure_ascii=False, indent=2)}"
                )
            ),
        ]
    )

    raw_text = response.content
    parsed = extract_json(raw_text)

    try:
        return OrchestratorDecision(**parsed)
    except ValidationError as e:
        raise ValueError(
            f"LLM returned invalid OrchestratorDecision.\n"
            f"Validation error: {e}\n"
            f"Raw response: {raw_text}"
        )

## Добавляем агента для описания